<a href="https://colab.research.google.com/github/ZoubirCHATTI/ai_assistant/blob/main/app.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
!pip install azure

  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (setup.py) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.


In [4]:
import streamlit as st
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from azure.storage.blob import BlobServiceClient
from langchain_experimental.agents.agent_toolkits import create_pandas_dataframe_agent
from langchain.agents import AgentExecutor, create_react_agent
from langchain_mistralai import ChartMistralAI
from langchain.tools import Tool
from langchain import hub
import io
import traceback
import requests

ModuleNotFoundError: No module named 'azure'

In [ ]:
st.set_page_config(page_title="Train Data Assistant", layout="wide")
st.title("🚆 Assistant IA - Analyse Ferroviaire")
try:
  AZURE_CONN=st.secrets["AZURE_STORAGE_CONNECTION_STRING"]
  MISTRAL_KEY = st.secrets["MISTRAL_API_KEY"]
except KeyError:
  st.error("⚠️ Clés de configuration manquantes dans les Secrets Streamlit.")
  st.info("Veuillez configurer AZURE_STORAGE_CONNECTION_STRING et MISTRAL_API_KEY.")
  st.stop()

In [ ]:
@st.cache_data(show_spinner="Connexion à azure et chargement des données...")
def load_data_from_azure():
  try:
    service_client = BlobServiceClient.from_connection_string(AZURE_CONN)
    blob_client= service_client.get_blob_client(container="ztacontainer", blob="otp_transformed.xlsx")
    stream = blob_client.download_blob().readall()
    data=pd.read_excel(io.BytesIO(stream))
    return data
  except Exception as e:
    st.error("Erreur de connexion ou de chargement depuis Azure : {e}")


In [ ]:
df= load_data_from_azure()
if len(df) > 10000:
  df_sample = df.sample(n=10000, random_state=42)
  st.warning("Utilisation d'un échantillon de 10,000 lignes (sur {len(df)}) pour accélérer l'analyse.")
else:
  df_sample = df

In [ ]:
def create_ferroviaire_tools(dataframe):
  @Tool

  def ponctualite_par_direction(direction:str)-> str:
    """
    Calcule le taux de ponctualité pour une direction spécifique ('N' ou 'S').
    Utilise cet outil pour répondre à des questions comme 'quelle est la ponctualité de la direction N ?'.
     """
    if direction.upper() not in ["N", "S"]:
      return "Erreur : La direction doit être 'N' ou 'S'."
    df_dir = dataframe[dataframe["direcion"].str.upper() == direction.upper()]
    if df_dir.empty:
      return "Aucune donnée trouvée pour la direction {direction}."
    total_train = len(df_dir)
    trains_a_l_heure = len(df_dir[df_dir["delay_minutes"]<=5])
    tax_ponctualite = (trains_a_l_heure/total_train)*100 if total_train > 0 else 0
    return (f"""Pour la direction {direction.upper()} : \n"
            f"- {total_trains} trains au total.\n"
            f"- {trains_a_l_heure} trains à l'heure.\n"
            f"- Taux de ponctualité : {taux_ponctualite:.2f}%.""")

  @Tool
  def heures_avec_le_plus_de_retards() -> str:
    """
    Identifie les 3 heures de la journée où il y a le plus de trains en retard.
    Utilise cet outil pour des questions comme 'quelles sont les pires heures ?' ou 'quand y a-t-il le plus de retards ?'.
    """
    df_retards = dataframe[dataframe['delay_minutes'] < 30]
    if df_retards.empty:
      return "Bonne nouvelle ! Aucun train en retard n'a été trouvé."

      top_heures = df_retards['hour'].value_counts().nlargest(3)
      result = "Les 3 heures avec le plus de retards sont :\n"
      for heure, count in top_heures.items():
        result += f"- {heure}h00 : {count} trains en retard.\n"
        return result

  @Tool
  def trains_les_plus_critiques() -> str:
    """
    Liste les 5 trains ayant accumulé le plus de retard en minutes.
    Utile pour identifier les 'pires trains' ou les 'retards les plus longs'.
     """
    top_trains = dataframe.sort_values(by='delay_minutes', ascending=False).nlargest(5, 'delay_minutes')
    result = "Les 5 trains avec les plus longs retards sont :\n"
    for _, row in top_trains.iterrows():
      result += f"- Train ID {row['train_id']} : {row['delay_minutes']} minutes de retard le {row['date']}.\n"
      return result

    return [ponctualite_par_direction, heures_avec_le_plus_de_retards, trains_les_plus_critiques]

In [ ]:
llm =ChartMistralAI(model="misltral-large-latest",
                    MISTRAL_API_KEY=MISTRAL_KEY, temperature=0)
pandas_agent = create_pandas_dataframe_agent(
    llm=llm,
    df_sample,
    verbose=True
    allow_dangerous_code= True,
    handle_parsing_errors = True
)
pandas_tool=Tool(
    name="Analyseur de dataframe Pandas",
    func= pandas_agent.invoke,
    description ="""
    Utilise cet outil pour toute question générale sur les données des trains.
    Il peut répondre à des questions sur les statistiques, filtrer des données, et même créer des graphiques (bar charts, line charts, etc.).
    Exemples de questions : 'combien de trains au total ?', 'fais-moi un bar chart des retards par jour de la semaine', 'quel est le retard moyen ?'.
    """
)

In [ ]:
outils_personalisés = create_ferroviare_tools(df_sample)
tous_les_outils = outils_personalisés + [pandas_tool]

In [ ]:
prompt = hub.pull("hwchase17/react")
agent_final = create_react_agent(llm, tous_les_outils, prompt)
 agent_executor = AgentExecutor(agent=agent_final, tools=tous_les_outils, verbose=True, handle_parsing_errors=True)

    # --- INTERFACE UTILISATEUR ---
    st.info("Assistant prêt ! Posez une question sur vos données.")
    st.write("Exemples de questions :")
    st.markdown("""
    - *Quelle est la ponctualité pour la direction S ?* (utilise un outil métier)
    - *Quelles sont les heures avec le plus de retards ?* (utilise un outil métier)
    - *Montre-moi un bar chart des retards par jour de la semaine.* (utilise l'analyseur Pandas)
    - *Combien y a-t-il de trains uniques ?* (utilise l'analyseur Pandas)
    """)

    query = st.text_input("Posez votre question ici :", key="main_query")

    if query:
        with st.spinner("L'IA réfléchit et choisit le meilleur outil..."):
            response = None
            try:
                # On invoque l'agent orchestrateur
                response = agent_executor.invoke({"input": query})

                st.markdown("#### Réponse de l'assistant :")
                st.write(response.get("output", "Désolé, je n'ai pas pu formuler de réponse claire."))

            except requests.exceptions.HTTPError as e:
                if e.response.status_code == 429:
                    st.error("⏳ Erreur 429 : Trop de requêtes envoyées à l'API Mistral. Veuillez attendre un moment avant de réessayer.")
                else:
                    st.error(f"Erreur de connexion avec l'API : {e}")
            except Exception as e:
                st.error("Oups, une erreur inattendue est survenue.")
                st.code(traceback.format_exc())

            # Afficher le graphique s'il en a généré un via l'agent Pandas
            if plt.get_fignums():
                st.success("Un graphique a été généré :")
                st.pyplot(plt.gcf())
                plt.clf() # Nettoyer la figure pour la prochaine requête

else:
    st.warning("Impossible de charger le DataFrame. L'application ne peut pas démarrer.")
    st.info("Vérifiez votre chaîne de connexion Azure et le nom du conteneur/fichier.")